# NGIML Single Image Inference

In [ ]:
import subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/juhenes/ngiml"
REPO_BRANCH = "FurtherEnhancement"
REPO_DIR = Path("/content/ngiml")

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", REPO_BRANCH], check=True)
else:
    subprocess.run([
        "git",
        "clone",
        "--branch",
        REPO_BRANCH,
        "--single-branch",
        REPO_URL,
        str(REPO_DIR),
    ], check=True)

sys.path.insert(0, str(REPO_DIR))
print(f"Repo ready at {REPO_DIR} on branch {REPO_BRANCH}")

In [ ]:
import io
from pathlib import Path

import numpy as np
import torch
from PIL import Image


from tools.infer_helpers import (
    load_model_from_checkpoint,
    run_multi_strategy_inference,
    sweep_checkpoint_inference_for_image,
)

from google.colab import drive, files

In [ ]:
# Paths
USE_COLAB_DRIVE = True
DRIVE_MOUNT = '/content/drive'
RUN_OUTPUT_DIR = Path('/content/drive/MyDrive/ngiml')
CHECKPOINT_DIR = RUN_OUTPUT_DIR / 'checkpoints'

# Inference parameters
NORMALIZATION_MODE = 'imagenet'
INFER_SIZE = 448
TILE_SIZE = 448
TILE_OVERLAP = 0.5
TILE_BATCH_SIZE = 16

STRATEGIES = [
    'direct',
    'sliding_window',
    'resize_keep_aspect_center_crop',
    'center_crop',
    'resize',
]

# Threshold selection
THRESHOLD_STRATEGY = 'fixed'  # one of: 'fixed', 'val_threshold'
FIXED_THRESHOLD = 0.5

# Mode selection
SWEEP_ALL_CHECKPOINTS = False
SWEEP_STRATEGY = 'direct'
SWEEP_SHOW_PLOTS = True
SWEEP_PLOT_MAX_CHECKPOINTS = None

drive.mount(DRIVE_MOUNT)

if THRESHOLD_STRATEGY not in {'fixed', 'val_threshold'}:
    raise ValueError("THRESHOLD_STRATEGY must be 'fixed' or 'val_threshold'")
THRESHOLD = float(FIXED_THRESHOLD) if THRESHOLD_STRATEGY == 'fixed' else None

if not CHECKPOINT_DIR.exists():
    raise FileNotFoundError(f'Checkpoint directory not found: {CHECKPOINT_DIR}')

print('Checkpoint directory:', CHECKPOINT_DIR)
print('Mode:', 'sweep_all_checkpoints' if SWEEP_ALL_CHECKPOINTS else 'single_checkpoint')
print('Threshold strategy:', THRESHOLD_STRATEGY, '| fixed value:', FIXED_THRESHOLD)

In [ ]:
if files is None:
    raise RuntimeError('This notebook currently expects Colab files.upload(). Run in Colab.')

uploaded = files.upload()

first_name = next(iter(uploaded.keys()))
image_bytes = uploaded[first_name]
pil_image = Image.open(io.BytesIO(image_bytes)).convert('RGB')
rgb_np = np.asarray(pil_image).astype(np.float32) / 255.0
rgb = torch.from_numpy(rgb_np).permute(2, 0, 1).contiguous()

print({'file': first_name, 'shape': tuple(rgb.shape[-2:])})

In [ ]:
if SWEEP_ALL_CHECKPOINTS:
    _ = sweep_checkpoint_inference_for_image(
        checkpoint_dir=CHECKPOINT_DIR,
        image=rgb,
        normalization_mode=NORMALIZATION_MODE,
        strategy=SWEEP_STRATEGY,
        threshold=THRESHOLD,
        infer_size=INFER_SIZE,
        tile_size=TILE_SIZE,
        tile_overlap=TILE_OVERLAP,
        tile_batch_size=TILE_BATCH_SIZE,
        show_progress=True,
        show_plot=SWEEP_SHOW_PLOTS,
        plot_max_checkpoints=SWEEP_PLOT_MAX_CHECKPOINTS,
    )

else:
    checkpoint_path = CHECKPOINT_DIR / 'best_checkpoint.pt'
    if not checkpoint_path.exists():
        epoch_ckpts = sorted(CHECKPOINT_DIR.glob('checkpoint_epoch_*.pt'), key=lambda p: p.stat().st_mtime)
        if not epoch_ckpts:
            raise FileNotFoundError(f'No checkpoint found in {CHECKPOINT_DIR}')
        checkpoint_path = epoch_ckpts[-1]

    model, device, ckpt_info = load_model_from_checkpoint(checkpoint_path)
    print('Using checkpoint:', checkpoint_path)
    print('Device:', device)
    print('Checkpoint info:', ckpt_info)

    results = run_multi_strategy_inference(
        model=model,
        image=rgb,
        device=device,
        normalization_mode=NORMALIZATION_MODE,
        threshold=THRESHOLD,
        infer_size=INFER_SIZE,
        tile_size=TILE_SIZE,
        tile_overlap=TILE_OVERLAP,
        tile_batch_size=TILE_BATCH_SIZE,
        strategies=STRATEGIES,
        show_plot=True,
        show_progress=True,
    )

    print({
        'mode': 'single_checkpoint',
        'normalization_mode': results['normalization_mode'],
        'threshold_strategy': THRESHOLD_STRATEGY,
        'threshold': results['threshold'],
        'strategies': results['strategies'],
    })
    print('Per-strategy summary:', results['summary'])